# 05 · Evaluación y análisis de errores

Se distinguen evidencia histórica de test, OCR aislado y pruebas funcionales. La ejecución no vuelve a usar test para elegir parámetros.

In [1]:
from pathlib import Path
import sys, json, csv, hashlib
from html import escape
from IPython.display import display, Markdown, HTML, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/paths.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
def load(relative):
    return json.loads((ROOT / relative).read_text(encoding="utf-8"))
def table(rows):
    if not rows:
        return display(Markdown("Sin filas."))
    keys = list(rows[0])
    head = "".join("<th>" + escape(str(k)) + "</th>" for k in keys)
    body = "".join("<tr>" + "".join("<td>" + escape(str(r.get(k, ""))) + "</td>" for k in keys) + "</tr>" for r in rows)
    display(HTML("<table><thead><tr>"+head+"</tr></thead><tbody>"+body+"</tbody></table>"))
print("Raíz del proyecto localizada. Se utilizan evidencias reales del repositorio.")


Raíz del proyecto localizada. Se utilizan evidencias reales del repositorio.


In [2]:
det=load("results/evaluation/detector_test.json")
table([{"métrica":k,"valor":v} for k,v in det["metrics"].items()])
print("Muestra del detector:",det["images"],"imágenes y",det["instances"],"instancias")

métrica,valor
precision,0.9863840747674092
recall,0.9792429792429792
map50,0.9941499485109381
map50_95,0.8934497049812269


Muestra del detector: 734 imágenes y 819 instancias


El registro histórico del detector no guardó hash del checkpoint. No se asigna retrospectivamente el hash actual a esa ejecución sin evidencia adicional.

In [3]:
ocr=load("results/evaluation/metricas_ocr.json")
assert abs(ocr["exactitud_placa"]-ocr["placas_exactas"]/ocr["placas_evaluables"])<1e-12
assert abs(ocr["cer"]-ocr["errores_edicion"]/ocr["caracteres_referencia"])<1e-12
table([{k:ocr[k] for k in ["total_recortes","placas_evaluables","placas_exactas","exactitud_placa","cer","cobertura_transcripcion"]}])

total_recortes,placas_evaluables,placas_exactas,exactitud_placa,cer,cobertura_transcripcion
54,22,17,0.7727272727272727,0.04697986577181208,0.4074074074074074


In [4]:
with (ROOT/"results/evaluation/detalle_ocr.csv").open(encoding="utf-8-sig",newline="") as f:
    rows=list(csv.DictReader(f))
errors=[r for r in rows if r["estado_manual"]=="legible" and r["coincide"]=="False"]
accepted=sum(float(r["confianza_media"])>=.85 and min(json.loads(r["confianzas_caracter"]))>=.60 for r in errors)
print("Lecturas erróneas evaluables:",len(errors),"; superan ambas reglas de confianza:",accepted)
print("No se muestran matrículas adicionales: la evidencia detallada está en el CSV autorizado.")

Lecturas erróneas evaluables: 5 ; superan ambas reglas de confianza: 3
No se muestran matrículas adicionales: la evidencia detallada está en el CSV autorizado.


## Pruebas funcionales actuales

Los ejemplos comprueban integración, no precisión estadística.

In [5]:
smoke=load("results/reports/prueba_integracion_real.json")
table([{"caso":r["case"],"HTTP":r["http_status"],"estado":r["validation"]["status"],"esperado":r["matches_expected_status"]} for r in smoke["cases"]])
assert smoke["all_expected_statuses_observed"]

caso,HTTP,estado,esperado
coincidencia,200,coincide,True
inconsistencia,200,inconsistencia,True
sin_placa,200,sin_placa,True


Para repetir integración con pesos reales: `python -m tests.smoke_real`. Para pruebas unitarias: `python -m unittest discover -s tests -v`. Ambas son distintas de volver a evaluar todo test. Las exclusiones OCR, ausencia de negativos y dependencia posible por vehículo siguen vigentes.